# Anexo probabilístico

Este notebook define y ejecuta modelos probabilísticos sobre las observaciones
del libro XLSX. El texto contiene únicamente supuestos, estimandos y
matemática. Los datos, los diagnósticos y los resultados aparecen como
`DataFrame`, figuras o archivos posteriores regenerados durante la ejecución.


## Entorno y controlador

El anexo no carga resúmenes históricos, cadenas preservadas ni tablas de una
ejecución anterior. Cada modelo se construye desde la fuente observacional
actual y utiliza semillas deterministas derivadas de identificadores estables.


In [ ]:
from __future__ import annotations

from pathlib import Path

from IPython.display import display  # pyright: ignore[reportUnknownVariableType]

from festuca_analysis.annex import ProbabilisticAnnex

project_root = Path.cwd().resolve()
if not (project_root / "pyproject.toml").is_file():
    raise FileNotFoundError("Ejecute el notebook desde la raíz del proyecto Festuca.")

annex = ProbabilisticAnnex(
    project_root=project_root,
    draws=2000,
    tune=2000,
    chains=4,
    cores=None,
    target_accept=0.95,
    export_results=True,
    export_figures=True,
    figure_profile="thesis",
    print_figure_json=False,
)


## Fuente y estados de las variables

La carga aplica el mismo contrato que el estudio clásico: mediciones primitivas,
fórmulas del libro, estimaciones explícitas y transformaciones del análisis se
mantienen separadas. Las estimaciones del libro no sustituyen mediciones
faltantes en el análisis primario. Las cantidades reconstruidas se calculan de
nuevo desde sus componentes y se comparan con las columnas del libro solo con
fines de auditoría.


In [ ]:
configuration = annex.configuration()
qa = annex.load_data()
provenance = annex.source_provenance()
source_audit = annex.source_audit()
variable_lineage = annex.variable_lineage()

assert annex.data is not None
longitudinal_data = annex.data.longitudinal.copy()
harvest_data = annex.data.harvest.copy()

display(longitudinal_data)
display(harvest_data)


## Modelo robusto para rendimiento

Sea $y_i$ el rendimiento de una parcela y sea

$$
z_i=\frac{y_i-c}{s}
$$

una transformación numérica definida por centro $c$ y escala $s$. El modelo es

$$
z_i\sim t_\nu(\mu_i,\sigma),
$$

$$
\mu_i=\alpha+\delta E_i+h(T_i)^\top\gamma+b(B_i)^\top\eta.
$$

$E_i$ identifica el contraste promedio entre el grupo con N experimental y el
tratamiento de referencia. $h(T_i)$ es una base ortonormal y centrada para los
calendarios fertilizados; por construcción, sus coeficientes describen forma y
no alteran el contraste promedio. $b(B_i)$ codifica contrastes de bloque.

La regularización jerárquica es

$$
\gamma_k=\tau_\gamma\gamma_k^*,
\qquad
\gamma_k^*\sim\mathcal N(0,1),
\qquad
\tau_\gamma\sim\operatorname{HalfNormal}(s_\gamma).
$$

Como $c$ y $s$ se calculan con la respuesta observada, la simulación de priors
es una auditoría condicional de escala y no una predicción genuinamente previa
a observar datos.


In [ ]:
model_specification = annex.model_specification()
conditional_prior_audit = annex.conditional_prior_predictive()


## Estimandos posteriores y chequeos predictivos

Las medias latentes se transforman de vuelta a la escala original. Se resumen
contrastes planificados, el rango entre calendarios, probabilidades de rango
respecto de una grilla de márgenes y probabilidades de posición relativa. Un
rango posterior no es una prueba de equivalencia sin un margen sustantivo
preespecificado.

Los chequeos posteriores replican la respuesta y vuelven a calcular el análisis
convencional. Su objetivo es evaluar qué tan compatible es ese resumen con el
modelo, no reinterpretar el valor $p$ observado como probabilidad de una
hipótesis.


In [ ]:
yield_model_runs = annex.fit_yield_models()
yield_diagnostics = annex.yield_diagnostics()
yield_posterior_means = annex.yield_posterior_summaries()
yield_margin_sensitivity = annex.margin_sensitivity()
yield_posterior_predictive = annex.posterior_predictive_checks()
sector_pattern_comparison = annex.sector_pattern_comparison()


## Modelo probabilístico longitudinal

Para observación $t$ de la parcela $k$:

$$
z_{kt}\sim t_\nu(\mu_{kt},\sigma),
$$

$$
\mu_{kt}=x_{kt}^{\top}\beta+w_{kt}^{\top}\gamma+u_k,
\qquad
u_k\sim\mathcal N(0,\sigma_u^2).
$$

$x_{kt}$ contiene intercepto, bloque y efectos principales; $w_{kt}$ contiene
la interacción tratamiento por fecha. La interacción recibe regularización
jerárquica:

$$
\gamma_q=\tau_I\gamma_q^*,
\qquad
\gamma_q^*\sim\mathcal N(0,1).
$$

Para una respuesta log-transformada, la transformación inversa de la
localización describe un valor típico geométrico. No se presenta como media
aritmética marginal sin integrar explícitamente la distribución residual.


In [ ]:
longitudinal_model_runs = annex.fit_longitudinal_models()
longitudinal_diagnostics = annex.longitudinal_diagnostics()
longitudinal_trajectories = annex.longitudinal_summaries()


## Nulo de reconstrucción de componentes

La variable semillas estimadas por panoja comparte el número de panojas en su
denominador. Para evaluar si una correlación contiene información adicional a
esa identidad, se permuta el numerador estimado y se reconstruye

$$
\widehat{S/P}^{\,*}=\frac{\pi(\widehat S)}{n_p}.
$$

La distribución resultante es un nulo de reconstrucción, no un modelo completo
de la biología reproductiva.

No se introduce un segundo modelo latente para el INN: el índice es una
transformación determinista de biomasa, concentración de N y una curva crítica.
La incertidumbre apropiada debe propagarse desde esos componentes y desde la
curva, no tratar el índice calculado como una medición independiente.


In [ ]:
reconstruction_null = annex.reconstruction_null()


## Diagnóstico, síntesis y exportación

Solo los modelos que superan los criterios declarados de convergencia se marcan
como utilizables en la síntesis. Una fila no utilizable se conserva para que el
fallo sea visible. La comparación entre sectores describe diferencias de patrón
y no identifica un efecto causal de condición hídrica.


In [ ]:
probabilistic_synthesis = annex.synthesis()
export_manifest = annex.export_artifacts()
